In [5]:

# ==========================================
# FILE: VIUSLIZATIONS
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score
from scipy import interpolate

from collections import defaultdict
from typing import List, Optional
from IPython.display import FileLink, display



def plot_learning_curves(history_df, fold, output_dir):
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history_df['epoch'], history_df['train_loss'], label='Train Loss', color='blue', linewidth=2)
    plt.plot(history_df['epoch'], history_df['val_loss'], label='Val Loss', color='red', linewidth=2)
    plt.title(f'Fold {fold}: Loss', fontsize=14)
    plt.xlabel('Epochs', fontsize=12)
    plt.legend(); plt.grid(True, linestyle='--', alpha=0.7)
    
    plt.subplot(1, 2, 2)
    plt.plot(history_df['epoch'], history_df['train_acc'], label='Train Acc', color='blue', linewidth=2)
    plt.plot(history_df['epoch'], history_df['val_acc'], label='Val Acc', color='red', linewidth=2)
    plt.title(f'Fold {fold}: Accuracy', fontsize=14)
    plt.xlabel('Epochs', fontsize=12)
    plt.legend(); plt.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.savefig(output_dir / f"learning_curves_fold_{fold}.png", dpi=300)
    plt.close()

def plot_confusion_matrix(y_true, y_pred, classes, output_dir):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, annot_kws={"size": 12})
    plt.title('Test Set Confusion Matrix', fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(output_dir / 'confusion_matrix.png', dpi=300)
    plt.close()

In [6]:

# ==========================================
# FILE: PREPROCESSING 
# ==========================================
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score
from scipy import interpolate
import logging
import json
import shutil
from collections import defaultdict
from typing import List, Optional
from IPython.display import FileLink, display

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)-8s | %(message)s')
logger = logging.getLogger()

SAMPLING_FREQ_RAW = 50000 
TACH_COL = 0               
VIBRATION_COLS = [1, 2, 3] 

# Optimized for Spectral Analysis
ORDERS_PER_REV = 64        # Max frequency analyzed is 32X RPM (captures bearing harmonics)
REVOLUTIONS_PER_WINDOW = 8 # 8 revs gives an ultra-precise frequency resolution of 0.125 Orders
RANDOM_STATE = 42


class SpectralOrderPreprocessor:
    def __init__(self, orders_per_rev: int = ORDERS_PER_REV, revolutions: int = REVOLUTIONS_PER_WINDOW):
        self.orders_per_rev = orders_per_rev
        self.revolutions = revolutions
        self.window_size = orders_per_rev * revolutions
    
    def detect_tach_pulses(self, tach_signal: np.ndarray, sampling_freq: float) -> np.ndarray:
        threshold = np.mean(tach_signal) + 0.5 * np.std(tach_signal)
        binary = tach_signal > threshold
        min_samples = max(1, int(sampling_freq / 5000))
        rising_edges, last_edge = [], -min_samples
        for i in range(1, len(binary) - 1):
            if not binary[i-1] and binary[i] and binary[i+1]:
                if i - last_edge > min_samples:
                    rising_edges.append(int(i)) 
                    last_edge = i
        return np.array(rising_edges, dtype=np.int32)
    
    def resample_to_orders(self, vib_signal: np.ndarray, tach_pulses: np.ndarray) -> Optional[np.ndarray]:
        if len(tach_pulses) < self.revolutions + 1: return None
        start_idx, end_idx = int(tach_pulses[0]), int(tach_pulses[self.revolutions])
        if end_idx <= start_idx or (end_idx - start_idx) < (self.window_size * 0.3): return None
        
        original_indices = np.arange(start_idx, end_idx, dtype=np.float32)
        target_indices = np.linspace(start_idx, end_idx, self.window_size, dtype=np.float32)
        resampled = np.zeros((self.window_size, vib_signal.shape[1]), dtype=np.float32)
        
        for ax in range(vib_signal.shape[1]):
            try:
                f = interpolate.interp1d(original_indices, vib_signal[start_idx:end_idx, ax], kind='cubic', fill_value="extrapolate")
                resampled[:, ax] = f(target_indices)
            except: 
                return None
        return resampled

    def fit_transform(self, files: List[Path], labels: List[str], max_files_per_class: int = 150):
        logger.info(f"🔄 Extracting Order-Tracked FFT Spectra (with Real-World Normalization)...")
        all_spectra, all_labels, all_sources = [], [], []
        class_counts = defaultdict(int)
        
        hanning_win = np.hanning(self.window_size)[:, None]
        
        for file_path, class_name in zip(files, labels):
            if class_counts[class_name] >= max_files_per_class and class_name != "Normal": 
                continue
                
            try:
                df = pd.read_csv(file_path, header=None)
                raw_vib = df.values[:, VIBRATION_COLS].astype(np.float32)
                raw_tach = df.values[:, TACH_COL].astype(np.float32)
                pulses = self.detect_tach_pulses(raw_tach, SAMPLING_FREQ_RAW)
                
                # ⚠️ FRIEND'S FIX 1: 50% Overlap for Normal to prevent memorization
                if class_name == "Normal":
                    max_windows = 40  # Reduced from 75
                    pulse_step = max(1, self.revolutions // 2) # 50% overlap (advance by 4 revs)
                else:
                    max_windows = 15  
                    pulse_step = self.revolutions # 0% overlap (advance by 8 revs)
                
                win_count = 0
                start_pulse = 0
                
                while win_count < max_windows and (start_pulse + self.revolutions + 1 <= len(pulses)):
                    window = self.resample_to_orders(raw_vib, pulses[start_pulse:start_pulse + self.revolutions + 1])
                    
                    if window is not None:
                        windowed_signal = window * hanning_win
                        spectrum = np.abs(np.fft.rfft(windowed_signal, axis=0)) / self.window_size
                        
                        all_spectra.append(spectrum)
                        all_labels.append(class_name)
                        all_sources.append(f"{class_name}_{file_path.name}")
                        win_count += 1
                        
                    start_pulse += pulse_step 
                    
                class_counts[class_name] += 1
            except Exception: 
                continue
        
        X = np.array(all_spectra, dtype=np.float32)
        X_log = np.log1p(X * 1000) 
        
        # ⚠️ FRIEND'S FIX 2: Global Maximum Scaling (Preserves relative fault amplitude vs normal)
        global_max = np.max(X_log) + 1e-8
        X_scaled = X_log / global_max
        
        return X_scaled, np.array(all_labels), np.array(all_sources)

In [7]:
# ==========================================
# FILE: deep_maf_spectral.py (Kaggle Ready & Leakage-Free)
# ==========================================
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score
from scipy import interpolate
import logging
import json
import shutil
from collections import defaultdict
from typing import List, Optional
from IPython.display import FileLink, display

# from .preprocessing_deep import SpectralOrderPreprocessor

# from .viuslizations_deep import plot_learning_curves, plot_confusion_matrix

# export KAGGLE_API_TOKEN=KGAT_36c8636e1329b3ae57e66da6ba3d34bf
# kaggle competitions list

# ==================== CONFIGURATION ====================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(4)
os.environ['OMP_NUM_THREADS'] = '4'

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)-8s | %(message)s')
logger = logging.getLogger()
logger.info(f"🚀 Using compute device: {DEVICE}")

# Dataset Configuration
SAMPLING_FREQ_RAW = 50000 
TACH_COL = 0               
VIBRATION_COLS = [1, 2, 3] 

# Optimized for Spectral Analysis
ORDERS_PER_REV = 64        # Max frequency analyzed is 32X RPM (captures bearing harmonics)
REVOLUTIONS_PER_WINDOW = 8 # 8 revs gives an ultra-precise frequency resolution of 0.125 Orders
RANDOM_STATE = 42

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# ⚠️ KAGGLE DATASET PATH
from pathlib import Path

# This makes the path absolute relative to your current file
RAW_DATA_ROOT = Path.cwd().parent.parent.parent / "data" / "raw_mafulda"
# Or, if you prefer the relative path style:

MODEL_DIR = Path("models/mafaulda_spectral")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_SOURCES = {
    "Normal": {"root": "normal"},
    "Imbalance": {"root": "imbalance"},
    "Horiz_Misalign": {"root": "horizontal-misalignment"},
    "Vert_Misalign": {"root": "vertical-misalignment"},
    "Ball_Fault": {"root": "underhang", "subfolders":["ball_fault"]},
    "Outer_Race": {"root": "underhang", "subfolders":["outer_race"]}
}



# ==================== DATASET ====================
class SpectralDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, augment: bool = False):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
        self.augment = augment
    
    def __len__(self): 
        return len(self.X)
    
    def __getitem__(self, idx): 
        x = self.X[idx].clone()
        if self.augment:
            # 1. Random Gain (Amplitude shift)
            if torch.rand(1).item() > 0.5:
                x = x * torch.empty(1).uniform_(0.9, 1.1)
                
            # 2. ⚠️ FRIEND'S FIX 3: Spectral Masking (Forces model to look at whole spectrum)
            if torch.rand(1).item() > 0.5:
                mask_size = torch.randint(2, 6, (1,)).item() # mask 2 to 5 frequency bins
                start = torch.randint(0, x.shape[0] - mask_size, (1,)).item()
                x[start:start+mask_size, :] = 0
                
        return x, self.y[idx]

# ==================== MODEL ====================
import torch
import torch.nn as nn
import torch.nn.functional as F

class SEBlock(nn.Module):
    """Squeeze-and-Excitation block to learn axis/feature importance (Axis Interaction)"""
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction, bias=False)
        self.fc2 = nn.Linear(channels // reduction, channels, bias=False)

    def forward(self, x):
        b, c, _ = x.size()
        y = x.mean(dim=2)  # Global average pooling (Squeeze)
        y = F.relu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y)).view(b, c, 1) # Attention weights (Excitation)
        return x * y

class MultiScaleConv(nn.Module):
    """Captures both high-frequency impacts and low-frequency rotations (Long-range context)"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Large strides and kernels to quickly downsample the massive 50kHz signal
        self.conv_large = nn.Conv1d(in_channels, out_channels // 3, kernel_size=128, stride=8, padding=60)
        self.conv_mid = nn.Conv1d(in_channels, out_channels // 3, kernel_size=32, stride=8, padding=12)
        self.conv_small = nn.Conv1d(in_channels, out_channels // 3, kernel_size=7, stride=8, padding=3)
        self.proj = nn.Conv1d((out_channels // 3) * 3, out_channels, kernel_size=1)
        self.bn = nn.BatchNorm1d(out_channels)
        
    def forward(self, x):
        x1 = self.conv_large(x)
        x2 = self.conv_mid(x)
        x3 = self.conv_small(x)
        out = torch.cat([x1, x2, x3], dim=1)
        return F.relu(self.bn(self.proj(out)))

class AdvancedVibrationCNN(nn.Module):
    def __init__(self, input_channels: int = 3, num_classes: int = 6):
        super().__init__()
        
        # --- TIME DOMAIN BRANCH ---
        self.time_branch = nn.Sequential(
            MultiScaleConv(input_channels, 64),
            SEBlock(64, reduction=8),
            nn.MaxPool1d(4),
            
            # Dilated convolutions to expand receptive field without adding parameters
            nn.Conv1d(64, 128, kernel_size=15, padding=14, dilation=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            SEBlock(128, reduction=8),
            nn.MaxPool1d(4),
            
            nn.Conv1d(128, 256, kernel_size=7, padding=12, dilation=4),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            SEBlock(256, reduction=16),
            nn.AdaptiveMaxPool1d(8),  # Standardizes output length to 8
            nn.Dropout1d(0.2)
        )
        
        # --- FREQUENCY DOMAIN BRANCH ---
        self.freq_branch = nn.Sequential(
            nn.Conv1d(input_channels, 64, kernel_size=31, padding=15, stride=4),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            SEBlock(64, reduction=8),
            nn.MaxPool1d(4),
            
            nn.Conv1d(64, 128, kernel_size=15, padding=7, stride=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            SEBlock(128, reduction=8),
            nn.AdaptiveMaxPool1d(8),
            nn.Dropout1d(0.2)
        )
        
        self.flatten = nn.Flatten()
        
        # --- FUSION & CLASSIFIER ---
        self.classifier = nn.Sequential(
            nn.Linear((256 * 8) + (128 * 8), 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        """ Expects input x of shape (Batch, 3_Axes, Time_Samples) """
        
        # 1. Time Domain Feature Extraction
        t_feat = self.time_branch(x)
        t_feat = self.flatten(t_feat)
        
        # 2. Dynamic Frequency Domain (FFT) Processing
        # Compute Real FFT, take magnitude, and use Log1p to scale large spectral peaks
        fft_x = torch.fft.rfft(x, dim=2).abs() 
        fft_x = torch.log1p(fft_x) 
        
        f_feat = self.freq_branch(fft_x)
        f_feat = self.flatten(f_feat)
        
        # 3. Fuse Physics features and classify
        fused = torch.cat([t_feat, f_feat], dim=1)
        out = self.classifier(fused)
        
        return out

# ==================== TRAINING ====================
def train_pytorch_model(X: np.ndarray, y: np.ndarray, groups: np.ndarray, le: LabelEncoder):
    num_classes = len(le.classes_)
    gkf = GroupKFold(n_splits=5)
    best_auc, best_model = 0, None

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups), 1):
        logger.info(f"--- Starting Fold {fold}/5 ---")
        
        X_train_fold, y_train_fold = X[train_idx], y[train_idx]
        
        class_counts = np.bincount(y_train_fold)
        sample_weights = (1.0 / class_counts)[y_train_fold]
        sampler = torch.utils.data.WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
        
        train_loader = DataLoader(SpectralDataset(X_train_fold, y_train_fold, augment=True), batch_size=64, sampler=sampler, num_workers=4 if torch.cuda.is_available() else 0)
        val_loader = DataLoader(SpectralDataset(X[val_idx], y[val_idx], augment=False), batch_size=128, shuffle=False)
        
        model = AdvancedVibrationCNN(num_classes=num_classes).to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        
        optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
        
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)
        
        best_fold_auc, patience_counter, max_patience = 0, 0, 7
        fold_logs = []
        
        for epoch in range(1, 81): 
            model.train()
            train_loss, correct, total = 0, 0, 0
            for bX, by in train_loader:
                bX, by = bX.to(DEVICE), by.to(DEVICE)
                optimizer.zero_grad()
                out = model(bX)
                loss = criterion(out, by)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                train_loss += loss.item()
                correct += (torch.argmax(out, 1) == by).sum().item()
                total += by.size(0)
            
            train_acc = correct / total
            
            model.eval()
            val_loss, all_preds, all_probs, all_labels = 0, [], [], []
            with torch.no_grad():
                for bX, by in val_loader:
                    bX, by = bX.to(DEVICE), by.to(DEVICE)
                    out = model(bX)
                    val_loss += criterion(out, by).item()
                    all_probs.extend(torch.softmax(out, 1).cpu().numpy())
                    all_preds.extend(torch.argmax(out, 1).cpu().numpy())
                    all_labels.extend(by.cpu().numpy())
            
            val_acc = accuracy_score(all_labels, all_preds)
            try: 
                val_auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
            except: 
                val_auc = val_acc
            
            # ⚠️ Step the Plateau Scheduler using validation loss
            scheduler.step(val_loss / len(val_loader))
            
            fold_logs.append({'epoch': epoch, 'train_loss': train_loss/len(train_loader), 'train_acc': train_acc,
                              'val_loss': val_loss/len(val_loader), 'val_acc': val_acc, 'val_auc': val_auc})
            
            if val_auc > best_fold_auc:
                best_fold_auc = val_auc
                patience_counter = 0
                torch.save(model.state_dict(), MODEL_DIR / f"best_fold_{fold}.pt")
            else: 
                patience_counter += 1
            
            if patience_counter >= max_patience: 
                logger.info(f"Early stopping at epoch {epoch}")
                break
                
        plot_learning_curves(pd.DataFrame(fold_logs), fold, RESULTS_DIR)
        model.load_state_dict(torch.load(MODEL_DIR / f"best_fold_{fold}.pt"))
        if best_fold_auc > best_auc:
            best_auc = best_fold_auc
            best_model = model
            torch.save(best_model.state_dict(), MODEL_DIR / "best_model.pt")
            
        del model; gc.collect()
    return best_model

# ==================== MAIN ====================
if __name__ == "__main__":
    logger.info("=== Starting Spectral Deep Learning Pipeline ===")
    base = Path(RAW_DATA_ROOT)
    all_files, all_labels = [], []
    
    for class_name, config in DATA_SOURCES.items():
        target_dir = base / config['root']
        if target_dir.exists():
            if "subfolders" in config:
                for sub in config['subfolders']:
                    if (sub_dir := target_dir / sub).exists():
                        files = list(sub_dir.rglob("*.csv"))
                        all_files.extend(files)
                        all_labels.extend([class_name]*len(files))
            else:
                files = list(target_dir.rglob("*.csv"))
                all_files.extend(files)
                all_labels.extend([class_name]*len(files))
    
    if not all_files:
        logger.error(f"❌ Dataset not found at {RAW_DATA_ROOT}!")
    else:
        preprocessor = SpectralOrderPreprocessor()
        X, y, sources = preprocessor.fit_transform(all_files, all_labels, max_files_per_class=150)
    
        le = LabelEncoder()
        y_encoded = le.fit_transform(y)
        
        logger.info("🔪 Performing Group-based Train/Test Split...")
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
        train_val_idx, test_idx = next(gss.split(X, y_encoded, groups=sources))
        
        X_train_val, y_train_val = X[train_val_idx], y_encoded[train_val_idx]
        sources_train_val = sources[train_val_idx]
        X_test, y_test = X[test_idx], y_encoded[test_idx]
        
        best_model = train_pytorch_model(X_train_val, y_train_val, sources_train_val, le)
        
        logger.info("🧪 Evaluating on strict held-out test set...")
        best_model.eval()
        
        test_loader = DataLoader(SpectralDataset(X_test, y_test, augment=False), batch_size=128, shuffle=False)
        
        all_preds, all_probs, all_labels = [], [], []
        with torch.no_grad():
            for bX, by in test_loader:
                bX, by = bX.to(DEVICE), by.to(DEVICE)
                out = best_model(bX)
                probs = torch.softmax(out, dim=1)
                all_probs.extend(probs.cpu().numpy())
                all_preds.extend(torch.argmax(probs, dim=1).cpu().numpy())
                all_labels.extend(by.cpu().numpy())
                
        all_preds, all_labels, all_probs = np.array(all_preds), np.array(all_labels), np.array(all_probs)
        class_names = le.classes_
        
        plot_confusion_matrix(all_labels, all_preds, class_names, RESULTS_DIR)
        
        report_dict = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)
        report_df = pd.DataFrame(report_dict).transpose()
        report_df.to_csv(RESULTS_DIR / "classification_report.csv")
        
        print("\n" + "="*50 + "\nFINAL TEST SET CLASSIFICATION REPORT\n" + "="*50)
        print(classification_report(all_labels, all_preds, target_names=class_names))
        
        pred_df = pd.DataFrame({"True_Label": le.inverse_transform(all_labels), "Pred_Label": le.inverse_transform(all_preds), "Correct": all_labels == all_preds})
        for i, class_name in enumerate(class_names): 
            pred_df[f"Prob_{class_name}"] = all_probs[:, i]
        pred_df.to_csv(RESULTS_DIR / "test_predictions_detailed.csv", index=False)
        
        test_acc = accuracy_score(all_labels, all_preds)
        try: 
            test_auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
        except: 
            test_auc = test_acc
            
        with open(RESULTS_DIR / "final_metrics.json", "w") as f: 
            json.dump({"Test_Accuracy": float(test_acc), "Test_ROC_AUC": float(test_auc)}, f, indent=4)
            
        logger.info(f"🎉 Pipeline Complete! Final Test Accuracy: {test_acc:.4f} | AUC: {test_auc:.4f}")

        zip_filename = "mafaulda_project_results"
        shutil.make_archive(zip_filename, 'zip', '/kaggle/working')
        display(FileLink(f'{zip_filename}.zip'))

ERROR:root:❌ Dataset not found at /data/raw_mafulda!
